# GDS-ParT: mathematics to a working classifier

This companion mirrors **chapters 0–15**. The original chapter-0/1 narration
and arithmetic cells are preserved below. Later chapters add executable
derivations, module boundaries, exercises, exact parameter counts and training.

Authority: `docs/GDS-ParT_Technical_Report.pdf` and its `.tex`; local production
code resolves implementation details. The independent model imports no production
modules. This is the repository's local GDS-ParT, **not** published ParT.
Use **Run All** with Python and PyTorch. Start in the repository root or this
companion directory (or the delivered course directory). See README for exact
PowerShell commands and caveats.
Synthetic results demonstrate software plumbing only, not physics performance.

In [1]:
from pathlib import Path
import sys, math, json
import torch
from torch import nn
from torch.nn import functional as F
companion = Path.cwd()
candidates = [companion, companion / "companion",
              companion / "docs" / "manim_course" / "companion"]
companion = next((p for p in candidates if (p / "gds_part_reconstruction.py").exists()), None)
assert companion is not None, "Start at repository, delivered course, or companion directory"
sys.path.insert(0, str(companion))
from gds_part_reconstruction import (
    Config, configurations, ParticleEmbedding, PairwiseEmbedding,
    RelationalGate, GatedAttention, GDSParTBlock, ClassAttentionBlock,
    GDSParT, pairwise_features, parameter_audit, shared_breakdown, cpu_demo)
from math_examples import worked_examples
torch.manual_seed(17)
torch.set_num_threads(2)
torch.set_printoptions(precision=6, sci_mode=False)
print("PyTorch", torch.__version__)

PyTorch 2.11.0+cu128


## Chapter 0: We will turn mathematics into a working particle classifier.
Welcome. We will build a particle classifier from mathematics you can inspect, not from unexplained boxes. You need school arithmetic and curiosity. We will introduce each new operation, calculate an example, and connect it to code. This first chapter establishes our destination.

Source: PDF pp. 6-7

## Chapter 0: You will identify the inputs, outputs, and two streams.
By the end of this chapter, you will identify what enters the model, what leaves it, and why two streams are needed. Understanding every internal operation comes later. For now, follow the moving particles and their relationships rather than memorising the architecture.

Source: PDF pp. 10-12

## Chapter 0: A jet contains particles, but receives one classification.
The report describes a jet as a collimated collection of reconstructed particles. Each dot represents one particle. The enclosing boundary represents one example. The prediction belongs to that entire example, not separately to every dot. A jet therefore enters as a collection.

Source: PDF p. 6

## Chapter 0: Each particle contributes seventeen measured or derived features.
The collection first becomes a table. Each row describes a particle, and each column holds one feature. There are seventeen input features in this implementation. Those seventeen observations are not seventeen trainable parameters. Learning adjusts the model's weights, not the recorded observations.

Source: PDF pp. 8-9, 33

## Chapter 0: Each pair contributes four descriptions of its relationship.
Now select two particles. Their relationship needs two addresses: the first particle and the second particle. The pair input contains four kinematic features. Notice the orange connection becoming an orange table entry. A relation belongs to a pair, rather than to one particle alone.

Source: PDF pp. 9-12

## Chapter 0: A shape records the meaning of every array axis.
We need names for sizes before displaying shapes. Capital B counts jets in a batch. N counts padded particle slots. D counts particle channels. D sub p counts relation channels. These letters describe dimensions. They are not additional measurements supplied by the detector.

Source: PDF p. 7

## Chapter 0: Particle states and pair relations form the two streams.
X names the particle states and stays blue. R names the pair representations and stays orange. X has one particle axis. R has two particle axes. In the reported configuration, their channel widths are one hundred twenty eight and sixty four. This is what dual stream means.

Source: PDF pp. 10-12

## Chapter 0: Particle states change while the relation representation is reused.
Watch the blue states move through successive blocks. The orange relation table stays in place and supplies information to each block. It is computed once per forward pass, not updated by a second recurrent transformer. Dual stream does not mean queries and keys.

Source: PDF p. 12

## Chapter 0: The reported model combines gates with additive pair bias.
The orange stream has two routes into particle attention. Red gates rescale query and key channels. Teal pair bias adjusts comparison scores. The reported model keeps both routes, so it is a hybrid. We will derive these routes separately before combining them.

Source: PDF pp. 11-12

## Chapter 0: The published model and local baseline are different systems.
Keep three comparisons distinct. Original Particle Transformer is the published architecture. Turning gates off here creates a local bias-only proxy, not an exact reproduction. The reported GDS model shares gates across blocks. The code's default instead uses separate, state-conditioned gates within each block.

Source: PDF pp. 7, 10

## Chapter 0: Every operation will connect a calculation to executable code.
Our route begins with algebra and vectors. We then build tensors, neural layers, attention, the two streams, and the complete classifier. For each important operation, we will compare a hand calculation with executable PyTorch. An output that merely has the right shape is not enough.

Source: PDF Appendices C-I; Sec. 16

## Chapter 0: Which representation needs two particle addresses?
Here is the first question. Which representation needs two particle addresses, X or R? Consider what one entry describes. Take a short moment to decide. Now follow the highlighted connection: it joins two particles, so the relation representation R needs both particle addresses.

Source: PDF Appendix I, exercise 10

## Chapter 0: Does the orange representation change after each particle block?
Second question: does the orange representation change after each particle block? Look at the diagram as the blue states advance. The orange table is reused. Its summaries can depend on the block configuration, but the relation tensor itself is not recurrently rewritten in this implementation.

Source: PDF Appendix I, exercise 10

## Chapter 0: We will explain every operation before assembling the model.
The destination is now clear. A collection of particle observations and pair observations produces one jet prediction. Blue particle states evolve; orange relations are reused. The reported model combines red gates and teal bias. Next, we learn the algebra needed to explain those operations rather than naming them.

Source: PDF pp. 10-12

## Chapter 1: Our model transforms particle observations and pair relationships.
Recall the destination. Blue states describe individual particles, and orange states describe their relationships. Before either becomes a neural representation, we need a language for calculations. This chapter introduces that language through a small number of examples that we can calculate completely.

Source: PDF pp. 12, 32-34

## Chapter 1: You will distinguish functions, vectors, and three different products.
By the end, you will evaluate a function, read an indexed sum, and distinguish elementwise, inner, and outer products. You will know the output shape before calculating. That habit prevents a common gating error: accidentally replacing a vector of channel scales with one number.

Source: PDF Appendices C-D

## Chapter 1: A variable names a value that can change.
Start with a simple problem: we want one rule that works for different inputs. The letter x names the input. The letter y names the output. Both are scalars here, meaning single numbers. The constants two and three specify the rule shown on screen.

Source: PDF C.1

## Chapter 1: Substitution replaces the input symbol with its chosen value.
Set x to four. Replace the input symbol with four without changing the rest of the rule. Multiply two by four to obtain eight, then add three to obtain eleven. The equation now tells us that the output y is eleven. Each step preserves equality.

Source: PDF C.1

## Chapter 1: A weight scales the input, while a bias shifts output.
Replace the fixed two by a weight w and the fixed three by a bias b. These are scalar parameters in this example. The weight controls scaling; the bias adds a shift. Training may change these parameters. Evaluating the rule with fixed parameters is not itself training.

Source: PDF C.1, C.5

## Chapter 1: An assignment updates a value rather than asserting equality.
There is a different meaning behind the left-pointing update arrow. It tells us to replace an old value with a new one. An equals sign asserts that two expressions have the same value. Later, residual blocks use updates. Do not read an update as an algebraic contradiction.

Source: PDF C.1

## Chapter 1: The code follows the same multiplication and addition.
Here is executable PyTorch for the scalar calculation. The tensor holds four. Multiplication produces eight, and addition produces eleven. The printed output is generated by running this code during the checks, not typed as an unverified claim. Compare every code line with its matching mathematical step.

Source: PDF C.1; computed code equivalent

## Chapter 1: A function maps an input to an output.
Now give a rule a name. The function f doubles its input and adds one. The function g squares its input. These names let us discuss the operations without rewriting them every time. Each function accepts a scalar and returns a scalar in this example.

Source: PDF C.2

## Chapter 1: Composition feeds one function's output into another function.
Apply f first. Its output becomes the input to g. We therefore replace the input of the squaring function by the entire expression two x plus one. The result is the square of that entire expression. The parentheses matter because both terms must be squared together.

Source: PDF C.2

## Chapter 1: Changing the order generally changes the resulting function.
Reverse the order. Squaring first gives x squared, and applying f then doubles that square and adds one. This is not the same expression as squaring two x plus one. A neural network is a composition too. Reordering its operations can change its behaviour.

Source: PDF C.2; algebraic consequence

## Chapter 1: The code preserves the order of the nested functions.
Using four as an illustrative input, the inner function f returns nine, and the outer function g returns eighty one. Reversing the order gives thirty three. These numbers are calculated from the report's functions. They are not trained-model measurements or experimental performance results.

Source: PDF C.1-C.2; derived illustrative outputs

## Chapter 1: A vector is an ordered list of numbers.
One scalar cannot store several observations at once. A vector collects numbers in an order. The report's example contains two, four, six, and eight. We write it vertically as a column vector. Its shape is four: there are four entries, not four separate examples.

Source: PDF D.1

## Chapter 1: An index selects an entry rather than multiplying it.
Lowercase c is the channel index. The subscript c selects one entry of x. When c is one, we select two. When c is three, we select six. The subscript does not multiply x by c. It is an address within the ordered list.

Source: PDF C.3, D.1

## Chapter 1: A summation adds the entries selected by its index.
The sigma symbol abbreviates repeated addition. Start the index at one and stop at D, the vector length. For this four-entry vector, the expression expands to two plus four plus six plus eight. Add the terms in sequence: six, twelve, then twenty.

Source: PDF C.3

## Chapter 1: The arithmetic mean divides the sum by the entry count.
An average uses both the sum and the count. The sum is twenty and the count is four, so the arithmetic mean is five. Do not divide by an unrelated dimension such as batch size. The denominator must count the entries included in this particular sum.

Source: PDF C.3

## Chapter 1: The reduction removes the axis that it sums.
The code stores the four entries in one tensor. Calling sum returns twenty, and mean returns five. Both outputs are scalars because the single entry axis has been reduced. The original tensor remains a vector. A reduction does not change the stored input in place.

Source: PDF C.3; PyTorch equivalent

## Chapter 1: Addition and scalar multiplication act on corresponding entries.
For vectors, addition combines matching positions. Scaling by a scalar multiplies every entry by the same number. The report writes these rules using a channel index. The output retains the vector's shape. Matching the positions is essential: the order defines which entries correspond.

Source: PDF D.1

## Chapter 1: The multiplication symbol must specify which operation you mean.
We now meet three operations that are often confused. The report uses a equal to one, two, and b equal to three, four. Both are vectors of length two. We will use the same inputs for all three products, so only the operation changes.

Source: PDF D.3

## Chapter 1: Elementwise multiplication pairs entries in the same position.
For the elementwise product, connect the first entry of a to the first entry of b. One times three gives three. Then connect the second entries. Two times four gives eight. Keep those two products as separate entries. Do not add them yet.

Source: PDF D.3

## Chapter 1: The Hadamard product returns a vector of matching products.
The output is the vector three, eight. Its shape is still two. The circled dot denotes this Hadamard product. This is the operation used when a query vector is multiplied by its gate. Each channel keeps its own scaled value, so the channel axis survives.

Source: PDF D.3

## Chapter 1: PyTorch uses an asterisk for elementwise tensor multiplication.
In PyTorch, a times b, written with an asterisk, performs this elementwise operation for these equal-shaped tensors. Running the code returns three and eight as two entries. A common mistake is to use matrix multiplication here. That would perform a different operation and change the output shape.

Source: PDF D.3; code gds_part_colab.py gate application

## Chapter 1: The inner product adds the matching products.
The dot product begins with exactly the same two pairwise multiplications. One times three is three, and two times four is eight. This time, add the results. Three plus eight gives eleven. The final output is one scalar, not a vector with two entries.

Source: PDF D.2-D.3

## Chapter 1: For these real vectors, dot product means inner product.
The report writes the dot product as a transposed vector multiplied by another vector. Transpose turns a column into a row. A one-by-two row times a two-by-one column produces one entry. Equivalently, sum the channelwise products using the index c from one to D.

Source: PDF D.2-D.3

## Chapter 1: The code confirms that the result is one scalar.
PyTorch dot returns eleven for these vectors. Summing their elementwise product gives the same answer. The empty shape shown by the check denotes a scalar tensor. Do not confuse that empty shape with an empty vector: this tensor contains one number, eleven.

Source: PDF D.2-D.3; computed code equivalent

## Chapter 1: A vector's length is the square root of its self-product.
The Euclidean norm gives the vector's length. Dot the vector with itself, then take a square root. The report also relates a dot product to the lengths of both vectors and the cosine of their angle. Therefore a dot product depends on magnitude as well as alignment.

Source: PDF D.2

## Chapter 1: The outer product multiplies every entry against every entry.
The outer product asks a different question. Instead of pairing only matching positions, pair every entry of a with every entry of b. The first row uses the first entry of a: one times three, then one times four. This row becomes three, four.

Source: PDF D.3

## Chapter 1: Each row fixes one entry from the first vector.
The second row uses the second entry of a. Two times three gives six, and two times four gives eight. Place those products in the second row. The completed output has two rows and two columns. Its four entries retain all pairwise combinations.

Source: PDF D.3

## Chapter 1: The outer product produces a matrix, not a scalar.
The notation is a column vector a multiplied by the row vector b transpose. The result is a two-by-two matrix. Compare it with the inner product: there, the transpose was on a and the output was scalar. The position of that transpose changes the computation.

Source: PDF D.3

## Chapter 1: The outer function computes the complete grid of products.
PyTorch outer constructs the same matrix. Its first row is three, four, and its second row is six, eight. Inspect the printed shape: two by two. This is why a successful multiplication alone is not enough. We must verify that it is the intended multiplication.

Source: PDF D.3; computed code equivalent

## Chapter 1: The same inputs can produce three different output shapes.
Place the three results together. Hadamard multiplication keeps two channelwise products. The inner product sums those products into eleven. The outer product stores four combinations in a matrix. For equal-length vectors, the Hadamard result is also the diagonal of this outer-product matrix.

Source: PDF D.3

## Chapter 1: Gates preserve channels before attention reduces them.
Return to the model map. A query vector q and its gate g Q have matching channel shapes. Their elementwise product produces a modified query with the same shape. Later, attention dots that query with a key vector to produce one score. Gating and scoring are different steps.

Source: PDF D.3, Sec. 7.4

## Chapter 1: Compute all three products before watching the solution.
Now solve the report's first exercise. Use a equal to two, minus one, and b equal to three, four. Predict the shape of each result before calculating. We will pause briefly, then work every multiplication and addition on screen. Pause longer yourself if you want.

Source: PDF Appendix I, exercise 1

## Chapter 1: Matching positions give six and negative four.
First calculate the Hadamard product. Two times three gives six in the first position. Minus one times four gives minus four in the second position. Keep the two numbers separate. The result is the vector six, minus four. Its shape is two.

Source: PDF Appendix I, answer 1

## Chapter 1: Adding the matching products gives the scalar two.
For the inner product, reuse the two matching products, six and minus four. Add them: six minus four equals two. The result is scalar. The negative second contribution reduces the total. This signed arithmetic matters later when we explain why larger gates need not increase attention.

Source: PDF Appendix I, answer 1; Sec. 8

## Chapter 1: All four combinations form a two-by-two matrix.
For the outer product, the first row is two times three and two times four, giving six and eight. The second row is minus one times three and minus one times four, giving minus three and minus four. None of these four entries is summed away.

Source: PDF Appendix I, answer 1

## Chapter 1: Running the exercise code reproduces all three answers.
The executable exercise confirms the vector, scalar, and matrix. Compare the minus signs carefully with the hand calculation. If your code produces only one number for the gate operation, you used a reduction too early. If it produces a matrix, you computed all combinations instead.

Source: PDF Appendix I, answer 1; computed code equivalent

## Chapter 1: Explain why reversing the functions changes their composition.
The second question is conceptual. Why does reversing f and g change the result? In one order, the addition happens before squaring and participates in the square. In the other order, the addition happens afterward. The diagram makes that difference visible without appealing to memorised terminology.

Source: PDF C.2

## Chapter 1: Choose the product that preserves a gate's channel vector.
Third question: which product should implement a per-channel gate? The output must retain one entry per channel, so use the Hadamard product. The inner product removes the channel axis, and the outer product introduces a second axis. The desired meaning determines the correct operation.

Source: PDF D.3

## Chapter 1: Shapes help distinguish operations before any code is written.
A variable names a value, a function transforms it, and composition chains transformations in a specified order. A vector stores ordered entries. Hadamard, inner, and outer products return a vector, scalar, and matrix respectively. Next, we extend this shape reasoning to batches, tensors, and masks.

Source: PDF Appendices C-E

## Executable calculation: scalar

In [2]:
import torch
x = torch.tensor(4.)
y = 2*x + 3
print(y)

tensor(11.)


## Executable calculation: function

In [3]:
import torch
x = torch.tensor(4.)
f = lambda x: 2*x + 1
g = lambda x: x*x
print(g(f(x)), f(g(x)))

tensor(81.) tensor(33.)


## Executable calculation: sum

In [4]:
import torch
x = torch.tensor([2.,4.,6.,8.])
print(x.sum(), x.mean())

tensor(20.) tensor(5.)


## Executable calculation: hadamard

In [5]:
import torch
a = torch.tensor([1.,2.])
b = torch.tensor([3.,4.])
print(a*b)

tensor([3., 8.])


## Executable calculation: dot

In [6]:
import torch
a = torch.tensor([1.,2.])
b = torch.tensor([3.,4.])
print(torch.dot(a,b))
print(torch.dot(a,b).shape)

tensor(11.)
torch.Size([])


## Executable calculation: outer

In [7]:
import torch
a = torch.tensor([1.,2.])
b = torch.tensor([3.,4.])
print(torch.outer(a,b))

tensor([[3., 4.],
        [6., 8.]])


## Executable calculation: exercise

In [8]:
import torch
a = torch.tensor([2.,-1.])
b = torch.tensor([3.,4.])
print(a*b)
print(torch.dot(a,b))
print(torch.outer(a,b))

tensor([ 6., -4.])
tensor(2.)
tensor([[ 6.,  8.],
        [-3., -4.]])


# Chapter 2: Tensors, shapes and masks


An axis names a kind of address, not a physical unit. Particle input F is
[B,N,17]; pair input U is [B,N,N,4]. The two N axes select query i and neighbour j.
A permutation must act on BOTH pair axes. Padding keeps tensor sizes rectangular;
`keep=True` identifies real particles. Broadcasting [B,1,N,1] excludes neighbour
j in a relation summary. Reducing the neighbour axis removes one N, not channels.

We use two jets with four slots; only three and two slots are real. Finite padding
may contain nonzero numbers. Never infer the mask from the feature value zero.

In [9]:
x = torch.randn(2, 4, 17)
u = torch.randn(2, 4, 4, 4)
keep = torch.tensor([[1,1,1,0], [1,1,0,0]], dtype=torch.bool)
print("F,U,keep", x.shape, u.shape, keep.shape)
weights = keep[:, None, :, None].float()
print("neighbour broadcast", weights.shape)
print("summary", ((u*weights).sum(2)/weights.sum(2)).shape)
assert keep.sum(1).tolist() == [3, 2]

F,U,keep torch.Size([2, 4, 17]) torch.Size([2, 4, 4, 4]) torch.Size([2, 4])
neighbour broadcast torch.Size([2, 1, 4, 1])
summary torch.Size([2, 4, 4])


### Check your understanding — worked answers

1. Which mask dimension addresses keys? In `[B,1,1,N]`, the final N.
2. Is an all-zero particle necessarily padding? No: only the explicit mask decides.
Summary: check axes before arithmetic, and permute both particle axes of U.

# Chapter 3: Logarithms, softmax and cross-entropy


Natural logarithms convert positive multiplicative scales to additive differences.
Softmax turns logits into probabilities: p_i=exp(s_i)/sum_j exp(s_j). Subtracting
the largest logit changes neither probabilities nor odds and improves stability.
The denominator normalizes one row at a time. Cross-entropy for target t is
`-log p_t`; feed **raw logits** to PyTorch cross_entropy, not probabilities.
For `[0,ln 2,ln 3]`, probabilities are `[1/6,2/6,3/6]`.

In [10]:
s = torch.tensor([0., math.log(2), math.log(3)], dtype=torch.float64)
p = s.softmax(0)
print("probabilities", p)
print("sum", p.sum().item())
print("CE target 2", F.cross_entropy(s[None], torch.tensor([2])).item())
torch.testing.assert_close(p, (s+100).softmax(0))
assert abs(p[2].item()-.5) < 1e-12

probabilities tensor([0.166667, 0.333333, 0.500000], dtype=torch.float64)
sum 1.0
CE target 2 0.6931471805599452


### Check your understanding — worked answers

1. Adding 100 to every logit changes what? Nothing in softmax.
2. What is loss for target index 2? `-ln(1/2)=ln 2≈0.693147`.
Summary: logits are not probabilities, and stable normalization matters.

# Chapter 4: Learning from examples


A classifier learns parameters from labelled training examples, not rules for
which physical identity a hidden channel must represent. The data split prevents
training loss from being mistaken for generalization. A Linear layer computes
X Wᵀ+b in PyTorch's row storage; W has [out,in] axes. GELU adds nonlinearity.
Accuracy measures decisions, while cross-entropy also measures confidence.
Our tiny examples are synthetic and do not estimate JetClass accuracy.

In [11]:
linear = nn.Linear(2, 3)
with torch.no_grad():
    linear.weight.copy_(torch.tensor([[1.,0.], [0.,1.], [1.,1.]]))
    linear.bias.copy_(torch.tensor([0.,1.,-1.]))
sample = torch.tensor([[2.,3.]])
print("affine output", linear(sample))
print("GELU", F.gelu(torch.tensor([-1.,0.,1.])))
print("parameters", sum(p.numel() for p in linear.parameters()))
assert sum(p.numel() for p in linear.parameters()) == 9

affine output tensor([[2., 4., 4.]], grad_fn=<AddmmBackward0>)
GELU tensor([-0.158655,  0.000000,  0.841345])
parameters 9


### Check your understanding — worked answers

1. Why nine parameters? Six matrix entries plus three biases.
2. Can zero training loss establish physics performance? No; use held-out,
representative data and appropriate metrics. Summary: capacity and evaluation
answer different questions.

# Chapter 5: Gradients and training


For a scalar parameter w, prediction is 3w+1 and squared error is (3w+1−5)².
At w=2, prediction 7 and loss 4 give derivative 2(7−5)3=12.
Gradient descent with learning rate 0.1 changes w to 0.8. Autograd stores
the computation graph and applies the chain rule backward. Zero gradients before
each new optimizer step unless deliberately accumulating micro-batches.
Training enables dropout; evaluation disables it but does not itself disable gradients.

In [12]:
w = torch.tensor(2., requires_grad=True)
prediction = 3*w+1
loss = (prediction-5).square()
loss.backward()
print("prediction, loss, gradient", prediction.item(), loss.item(), w.grad.item())
with torch.no_grad():
    w -= .1*w.grad
print("updated w", w.item())
assert abs(w.item()-.8) < 1e-6

prediction, loss, gradient 7.0 4.0 12.0
updated w 0.7999999523162842


### Check your understanding — worked answers

1. Why is the derivative 12, not 4? The chain rule multiplies
the outer derivative 2(7−5) by the inner derivative 3.
2. Why `no_grad` for a manual update? Parameter updates are not new model
operations whose graph we want to differentiate. Summary: loss → backward → step.

# Chapter 6: Jets and seventeen particle features


A jet is a set of reconstructed constituents receiving one class label. The
17 features are fixed-preprocessed observations: four log scales/fractions;
distance to jet axis; Δη, Δφ; charge; five identity flags; and four tracking
features. The cache uses `(raw−shift)*scale`, clips to [-5,5], then replaces
non-finite values. Shifts/scales are not learned. Four-vectors remain separate
float32 data, while cached constituent features use float16.

Class order is QCD,Hbb,Hcc,Hgg,H4q,Hqql,Zqq,Wqq,Tbqq,Tbl.
Track displacement can be informative but does not prove a flavour label.

In [13]:
pt, energy, jet_pt, jet_energy = 80., 82., 200., 220.
first_four = torch.tensor([
    .7*(math.log(pt)-1.7), .7*(math.log(energy)-2.),
    .7*(math.log(pt/jet_pt)+4.7), .7*(math.log(energy/jet_energy)+4.7)])
print("first four preprocessed features", first_four.clamp(-5,5))
print("feature count", 4+1+2+1+5+4)
assert 4+1+2+1+5+4 == 17

first four preprocessed features tensor([1.877419, 1.684703, 2.648597, 2.599164])
feature count 17


### Check your understanding — worked answers

1. Does a 128-dimensional embedding add 111 measurements? No, it
learns a representation from the same 17 inputs.
2. Are pair features derived from two 17-vectors? No; they use separate
four-momenta. Summary: distinguish observation, preprocessing and representation.

# Chapter 7: Four-vectors and pair features


Stored order is (px,py,pz,E). Compute pT, azimuth atan2(py,px), and rapidity
y=½ln((E+pz)/(E−pz)). Wrap Δφ into [-π,π), then
Δ=sqrt(Δy²+Δφ²), kT=min(pTi,pTj)Δ, z=min(pTi,pTj)/(pTi+pTj),
and m²=(Ei+Ej)²−|pi+pj|². The four input features are their natural logs.
Floors of 1e-8 match cache operations; self/padded pairs are multiplied by zero.
This is not robust NaN removal: NaN×0 remains NaN.

For massless pT=(80,30), Δy=.15 and Δφ=−.15, m²≈108.0002.
Massless rapidity equals pseudorapidity; do not assume equality for massive particles.

In [14]:
pt = torch.tensor([80.,30.])
y, phi = torch.tensor([.15,0.]), torch.tensor([-.15,0.])
vectors = torch.stack((pt*phi.cos(), pt*phi.sin(),
                       pt*y.sinh(), pt*y.cosh()), -1)[None]
pairs = pairwise_features(vectors, torch.ones(1,2,dtype=torch.bool))
print("pair log features", pairs[0,0,1])
torch.testing.assert_close(pairs[0,0,1],
    torch.tensor([-1.5505,1.8507,-1.2993,4.6821]), atol=1e-4, rtol=0)
torch.testing.assert_close(pairs, pairs.transpose(1,2))

pair log features tensor([-1.550546,  1.850651, -1.299283,  4.682134])


### Check your understanding — worked answers

1. Why z≤1/2? The smaller positive pT is at most half their sum.
2. Should arbitrary constituent-pair mass peak at a parent resonance? No:
the full decay system or appropriate subjets are needed. Summary: these are
pair descriptors, not unique reconstructed physical splittings.

# Chapter 8: Embeddings, LayerNorm and two streams


Particle embedding is LN17→Linear(17,D)→GELU→Linear(D,D)→Dropout.
Pair embedding is Linear(4,P)→GELU→Linear(P,P), without normalization/dropout.
LayerNorm normalizes channels independently per row with population variance
(divide by D, not D−1), epsilon 1e-5, and learned gamma/beta.
For [2,4,6,8], mean=5 and variance=5.

R is computed once and stays static. X changes through particle blocks.
Zero input pairs can become nonzero embedded relations because affine biases exist;
self pairs are not excluded from the masked neighbour mean.

In [15]:
v = torch.tensor([2.,4.,6.,8.])
gamma, beta = torch.tensor([1.,2.,.5,1.]), torch.full((4,),.1)
print("mean, population variance", v.mean().item(), v.var(unbiased=False).item())
print("LayerNorm", F.layer_norm(v, (4,), gamma, beta))
states = ParticleEmbedding(17,16,0)(x)
relations = PairwiseEmbedding(4,8)(u)
print("X,R", states.shape, relations.shape)

mean, population variance 5.0 5.0
LayerNorm tensor([-1.241639, -0.794426,  0.323607,  1.441640])
X,R torch.Size([2, 4, 16]) torch.Size([2, 4, 4, 8])


### Check your understanding — worked answers

1. Why two stream shapes? X has one particle address, R has two.
2. Is the pair embedding recomputed each block? No; only its use changes.
Summary: learned representations and their update rules are distinct.

# Chapter 9: Self-attention from arithmetic


Queries ask, keys provide comparison coordinates, and values carry the mixture.
For each head, score_ij=q_i·k_j/sqrt(d); softmax over j creates weights that sum
to one before dropout. Weighted values are then concatenated and linearly projected.
Q,K,V are separate learned affine projections, not different raw inputs.
With q=[1,0], keys the 2×2 identity and values [[2,0],[0,4]], d=2.

In [16]:
q, keys = torch.tensor([1.,0.]), torch.eye(2)
scores = keys @ q / math.sqrt(2)
probability = scores.softmax(0)
value = torch.tensor([[2.,0.],[0.,4.]])
print("scores", scores)
print("weights", probability)
print("weighted values", probability @ value)
assert abs(probability.sum().item()-1) < 1e-6

scores tensor([0.707107, 0.000000])
weights tensor([0.669762, 0.330238])
weighted values tensor([1.339523, 1.320954])


### Check your understanding — worked answers

1. Why divide by sqrt(d), not D? Each head's dot product has d terms.
2. Does attention softmax normalize channels? No, neighbouring key positions j.
Summary: the comparison and the value mixture are separate operations.

# Chapter 10: Pair bias and attention odds


The learned pair projection P→H supplies one scalar bias per head and pair.
Attention softmax uses score+bias. Odds obey p_j/p_k=exp(score_j−score_k),
so increasing only bias_j by one multiplies its odds by e.
Padding is a hard exclusion with −infinity, not a learned penalty.
The local model may use both additive bias and Q/K gates; these are not synonyms.

In [17]:
biased = (scores+torch.tensor([0.,1.])).softmax(0)
print("biased weights", biased)
print("odds multiplier", (biased[1]/biased[0]/(probability[1]/probability[0])).item())
masked = scores.masked_fill(torch.tensor([False,True]), -torch.inf).softmax(0)
print("one real key", masked)
bias = nn.Linear(8,2)(relations).permute(0,3,1,2)
print("pair bias", bias.shape)

biased weights tensor([0.427296, 0.572704])
odds multiplier 2.7182815074920654
one real key tensor([1., 0.])
pair bias torch.Size([2, 2, 4, 4])


### Check your understanding — worked answers

1. Which learned tensor has [B,H,N,N] axes? Additive pair bias.
2. Can one valid key get attention probability below one without dropout?
No, its row normalization is one. Summary: finite preference differs from exclusion.

# Chapter 11: Relational gates


Shared gates average R_ij over real neighbours j, then use independent
P→D→D MLPs for Q and K. The shared summaries/gates are reused across all blocks.
Conditioned gates instead form c_i=Linear(LN(X_i)), score each relation with
<R_ij,c_i>/sqrt(P), mask padded j and softmax before weighted summation.
Each particle block owns its conditioned gate.

Identity gates use 1+tanh with zero-initialized final affine layers, hence start
exactly at one for finite summaries. Sigmoid gates use (0,1) and are not identity.
Floating point saturation can reach range endpoints.

In [18]:
shared_gate = RelationalGate(8,16)
gq, gk = shared_gate(relations, keep)
print("exact identity", torch.equal(gq,torch.ones_like(gq)),
      torch.equal(gk,torch.ones_like(gk)))
conditioned_gate = RelationalGate(8,16,conditioned=True)
cgq, cgk = conditioned_gate(relations, keep, nn.LayerNorm(16)(states))
print("conditioned shapes", cgq.shape, cgk.shape)
print("sigmoid zero", torch.tensor(0.).sigmoid().item())

exact identity True True
conditioned shapes torch.Size([2, 4, 16]) torch.Size([2, 4, 16])
sigmoid zero 0.5


### Check your understanding — worked answers

1. Are query/key gate MLP weights shared with each other? No.
2. Do all v2 configurations use conditioned gates? No, the reported v2
uses shared mean gates; production defaults are conditioned.
Summary: activation, aggregation and sharing are separate switches.

# Chapter 12: Gate effects, gradients and expressivity


Gates multiply query/key channels BEFORE the dot product; values are ungated.
Halving a positive score flattens attention, but making a negative score less
negative raises its relative probability. Thus 'suppression' is not a universal
statement about final attention probability.

At identity initialization, gate last-layer weights are zero, so gradients to
earlier gate/context layers are initially zero; the last layers can still learn.
Relation summaries compress N neighbours to P channels: gates cannot encode an
arbitrary independent pair bias. Shared R is static, not a second self-attention stack.

In [19]:
print("half-query weights", (scores*.5).softmax(0))
print("negative logits before", torch.tensor([-2.,0.]).softmax(0))
print("negative logits after", torch.tensor([-1.,0.]).softmax(0))
gate = RelationalGate(8,16,conditioned=True)
a, b = gate(relations.detach(),keep,states.detach())
(a.sum()+b.sum()).backward()
print("context gradient", gate.ctx.weight.grad.abs().sum().item())
print("last gate gradient nonzero", bool(gate.query_gate[-1].weight.grad.abs().sum()>0))

half-query weights tensor([0.587479, 0.412521])
negative logits before tensor([0.119203, 0.880797])
negative logits after tensor([0.268941, 0.731059])
context gradient 0.0
last gate gradient nonzero True


### Check your understanding — worked answers

1. Does smaller query norm always lower every probability? No;
normalization and score signs matter.
2. Does zero early gradient mean identity gates never learn? No; last-layer
gradients open the earlier routes after updates. Summary: reason about derivatives
and complete normalized rows rather than a single channel in isolation.

# Chapter 13: The complete forward pass


Compute X,R once. For each pre-normalized block H=LN1(X):
X←X+Attention(H,Gate(R,H),bias), then X←X+FFN(LN2(X)).
FFN is D→ffn_dim→D with GELU and dropout.
Class pooling prepends a learned CLS token, normalizes [CLS;X], queries only
with CLS, and attends to CLS plus real particles. It has no relation gate or bias.
Only CLS updates; final LN and D→D→10 classifier produce logits.
Mean-pooling v1 instead applies final LN to particles BEFORE masked averaging.

Particle blocks are permutation equivariant, final logits invariant, provided
both pair axes and mask are permuted consistently and dropout is disabled.

In [20]:
cfg = Config(embed_dim=16, pairwise_embed_dim=8, ffn_dim=32,
             n_heads=2, n_layers=2, n_cls_layers=1, dropout=0)
model = GDSParT(cfg).eval()
logits, trace = model(x,u,keep,return_trace=True)
print(json.dumps(trace,indent=2))
order = torch.tensor([2,0,3,1])
permuted = model(x[:,order],u[:,order][:,:,order],keep[:,order])
torch.testing.assert_close(logits,permuted,atol=2e-6,rtol=2e-5)
print("logits permutation invariant within tolerance")

{
  "particles": [
    2,
    4,
    16
  ],
  "relations": [
    2,
    4,
    4,
    8
  ],
  "blocks": [
    [
      2,
      4,
      16
    ],
    [
      2,
      4,
      16
    ]
  ],
  "relation_evaluations": 1,
  "logits": [
    2,
    10
  ]
}
logits permutation invariant within tolerance


### Check your understanding — worked answers

1. Does CLS attention receive the pair bias? No.
2. Is the relation stream updated after a block? No.
Summary: the forward graph specifies where information is reused, changed and pooled.

# Chapter 14: Independent PyTorch reconstruction

Read `gds_part_reconstruction.py` top-to-bottom alongside these progressively
executable cells. Every module is independently implemented using PyTorch;
no production model is imported/wrapped. The source docstrings explain tensor
axes, formulas, zero initialization, masks, residual order and pooling.

The following short cells are the same sources captured in
`output/chapter_examples.json`. Tiny widths make CPU inspection quick; the
parameter audit instantiates the full report configurations. Counts come from
real modules and `sum(p.numel())`, not offsets or spare parameters.

### Imports

In [21]:
import torch
from gds_part_reconstruction import (
    Config, ParticleEmbedding,
    PairwiseEmbedding, RelationalGate,
    GatedAttention, GDSParTBlock,
    ClassAttentionBlock, GDSParT,
    parameter_audit, shared_breakdown)
torch.manual_seed(17)
torch.set_num_threads(2)

### Inputs

In [22]:
x = torch.randn(2, 4, 17)
u = torch.randn(2, 4, 4, 4)
keep = torch.tensor([[1, 1, 1, 0],
                     [1, 1, 0, 0]]).bool()
print(tuple(x.shape))
print(tuple(u.shape))
print(keep.sum(1).tolist())

(2, 4, 17)
(2, 4, 4, 4)
[3, 2]


### Particle

In [23]:
embed = ParticleEmbedding(17, 16, 0)
states = embed(x)
print(tuple(states.shape))

(2, 4, 16)


### Pair

In [24]:
pair_embed = PairwiseEmbedding(4, 8)
relations = pair_embed(u)
print(tuple(relations.shape))

(2, 4, 4, 8)


### Shared Gate

In [25]:
gate = RelationalGate(8, 16)
gq, gk = gate(relations, keep)
print(tuple(gq.shape), tuple(gk.shape))
print(torch.equal(gq, torch.ones_like(gq)))
print(torch.equal(gk, torch.ones_like(gk)))

(2, 4, 16) (2, 4, 16)
True
True


### Conditioned

In [26]:
gate = RelationalGate(8, 16,
                      conditioned=True)
norm = torch.nn.LayerNorm(16)
gq, gk = gate(relations, keep, norm(states))
print(tuple(gq.shape))
print(tuple(gate.ctx.weight.shape))

(2, 4, 16)
(8, 16)


### Bias

In [27]:
bias_layer = torch.nn.Linear(8, 2)
bias = bias_layer(relations)
bias = bias.permute(0, 3, 1, 2)
print(tuple(bias.shape))

(2, 2, 4, 4)


### Attention

In [28]:
attn = GatedAttention(16, 2, 0)
mixed, weights = attn(
    states, (gq, gk), bias, keep, True)
print(tuple(mixed.shape))
print(tuple(weights.shape))
print(weights[0, 0, 0, 3].item())

(2, 4, 16)
(2, 2, 4, 4)
0.0


### Block

In [29]:
cfg = Config(embed_dim=16, n_heads=2,
    pairwise_embed_dim=8, ffn_dim=32,
    n_layers=2, n_cls_layers=1, dropout=0)
block = GDSParTBlock(cfg)
updated = block(
    states, relations, None, bias, keep)
print(tuple(updated.shape))

(2, 4, 16)


### Cls

In [30]:
cls = torch.zeros(2, 1, 16)
pool = ClassAttentionBlock(cfg)
cls = pool(updated, cls, keep)
print(tuple(cls.shape))

(2, 1, 16)


### Logits

In [31]:
model = GDSParT(cfg).eval()
logits, trace = model(
    x, u, keep, return_trace=True)
print(tuple(logits.shape))
print(trace["relation_evaluations"])
print(trace["blocks"])

(2, 10)
1
[[2, 4, 16], [2, 4, 16]]


### Backward

In [32]:
labels = torch.tensor([0, 1])
loss = torch.nn.functional.cross_entropy(
    logits, labels)
loss.backward()
grad = model.classifier[-1].weight.grad
print(bool(torch.isfinite(grad).all()))
print(bool(grad.abs().sum() > 0))

True
True


### Counts

In [33]:
for name, count in parameter_audit().items():
    print(name, count)

v1 1280684
shared 2074420
conditioned 2488116
no_gates 2024756


### Breakdown

In [34]:
for name, count in shared_breakdown().items():
    print(name, count)
print("total", sum(shared_breakdown().values()))

particle_embedding 18850
pair_embedding 4480
gates 49664
bias 520
attention 528384
block_norms 4096
feed_forward 1053696
cls_token 128
cls_blocks 396544
final_norm 256
classifier 17802
total 2074420


### Check your understanding — worked answers

1. Why does v1 need six blocks? The report's historical configuration uses six;
production's four legacy operation switches alone leave eight. Six gives
1,280,684 parameters; eight would give 1,677,228.
2. Why can strict state-dictionary copying test independence? It checks every
actual parameter name/shape and then compares separate forward implementations.
It does not replace independent code with a production wrapper.

Shared audit: 18,850+4,480+49,664+520+528,384+4,096+1,053,696+
128+396,544+256+17,802 = **2,074,420**. Conditioned gates add one context
projection per block and have no shared gate module.
Summary: build, inspect, test, then train — counting alone cannot prove correctness.

# Chapter 15: Experiments, workflow and limitations


This last cell overfits eight fixed synthetic examples on CPU. Loss falling and
100% training accuracy establish that gradients/updates connect, not JetClass
performance. Use held-out evaluation, fixed split/preprocessing, independent seeds,
and matched budgets to compare architectures. The report's trained shared v2
is different from the per-layer conditioned default and from published ParT.
The reported v1→v2 improvement changes multiple choices, not just gates.

Run `tests.py` for shapes, exact initialization, block equivariance, invariant
logits, padding, strict production state-dict/logit parity, physics features,
numeric caveats, all parameter counts and overfitting. Run `train_demo.py` for
the standalone loss trajectory.

Finite padding is required. Masking does not remove NaNs or arbitrary overflow.
Explicit attention stores N×N weights; production SDPA/checkpointing are more
memory efficient. Nonempty jets are enforced by this teaching implementation.
Float32 reordering means symmetry/parity tests use tolerances, not exact equality.

In [35]:
report = cpu_demo(140)
print(json.dumps(report,indent=2))
assert report["final_loss"] < report["history"][0]["loss"]
assert report["training_accuracy"] == 1.
print("Synthetic overfit passed; no physics-performance claim.")

{
  "kind": "synthetic memorisation only",
  "steps": 140,
  "history": [
    {
      "step": 0,
      "loss": 0.7330597639083862
    },
    {
      "step": 20,
      "loss": 0.0016619269736111164
    },
    {
      "step": 40,
      "loss": 1.9222470655222423e-06
    },
    {
      "step": 60,
      "loss": 5.662438979925355e-07
    },
    {
      "step": 80,
      "loss": 5.215404144109925e-07
    },
    {
      "step": 100,
      "loss": 5.960462203802308e-07
    },
    {
      "step": 120,
      "loss": 6.407496471183549e-07
    }
  ],
  "final_loss": 7.152553962441743e-07,
  "training_accuracy": 1.0
}
Synthetic overfit passed; no physics-performance claim.


### Check your understanding — worked answers

1. Does matching parameter count prove the architecture matches?
No; also check state dictionaries, forward values, masking, symmetries and gradients.
2. Is a 100% toy training accuracy a benchmark? No; the examples are memorized
and have no independent evaluation set. Summary: transparent assumptions and
reproducible tests are necessary before drawing experimental conclusions.